Step Functions and EventBridge are AWS's orchestration and event routing services. Step Functions coordinates multi-step workflows — sequencing Lambda functions, AWS services, and human approvals into reliable state machines. EventBridge is a serverless event bus that routes events from AWS services, SaaS applications, and custom sources to targets based on rules. Together they are the backbone of event-driven and workflow-driven serverless architectures on AWS.

## Why Step Functions?

Without Step Functions, multi-step workflows are typically implemented by chaining Lambda functions — Lambda A calls Lambda B calls Lambda C. The problems:
- **No visibility** into which step failed or where execution is
- **Retry logic** must be written in every function
- **Error handling** is scattered and inconsistent
- **Long-running workflows** hit Lambda's 15-minute timeout

Step Functions moves orchestration logic **out of application code** and into a managed state machine:

```text
Without SFN:                    With Step Functions:
Lambda A                        State Machine
  └─ calls Lambda B               ├─ State: ValidateOrder  (Lambda)
       └─ calls Lambda C          ├─ State: ChargePayment  (Lambda)
            └─ calls Lambda D     ├─ State: SendEmail      (SES)
  (error handling in each)        └─ State: NotifyShipping (SNS)
                                  (retry, catch, timeout built-in)
```

## State Machine Concepts

A Step Functions **state machine** is defined in **Amazon States Language (ASL)** — a JSON document describing states and transitions.

### State types

| State | Purpose |
|---|---|
| **Task** | Invoke a Lambda, call an AWS API, run an ECS task, send to SQS, etc. |
| **Choice** | Branch based on input data (if/else) |
| **Wait** | Pause for a fixed duration or until a timestamp |
| **Parallel** | Execute multiple branches simultaneously; wait for all to complete |
| **Map** | Iterate over an array and apply a sub-workflow to each item |
| **Pass** | Pass input to output unchanged (useful for injecting data) |
| **Succeed** | End execution successfully |
| **Fail** | End execution with an error |

### Execution flow
```text
Start
  │
  ▼
ValidateOrder (Task: Lambda)
  │
  ▼
IsInventoryAvailable? (Choice)
  ├── Yes ──▶ ProcessPayment (Task: Lambda)
  │               │
  │           Parallel ──▶ SendConfirmation (Task: SES)
  │               └──▶ ReserveInventory   (Task: DynamoDB)
  │
  └── No  ──▶ NotifyOutOfStock (Task: SNS) ──▶ Fail
```

## Standard vs Express Workflows

| | Standard Workflow | Express Workflow |
|---|---|---|
| **Max duration** | 1 year | 5 minutes |
| **Execution model** | Exactly-once — each state executes once | At-least-once (async) or at-most-once (sync) |
| **Pricing** | Per state transition | Per execution duration + requests |
| **Execution history** | Stored in Step Functions (viewable in console) | Sent to CloudWatch Logs |
| **Throughput** | Up to 2,000 executions/s | Very high throughput (100,000+/s) |
| **Use case** | Long-running, auditable business workflows | High-volume, short-duration event processing |

**Standard:** order fulfilment, data pipelines, human-in-the-loop approvals, anything needing auditability and exactly-once execution.

**Express:** IoT ingestion, high-throughput microservice orchestration, streaming data transformation, short-lived automation.

> **Exam tip:** Long-running + auditable → Standard. High-throughput + short → Express.

## Error Handling: Retry and Catch

### Retry
Automatically retry a failed Task state with exponential backoff:

```json
"Retry": [{
  "ErrorEquals": ["Lambda.ServiceException", "Lambda.TooManyRequestsException"],
  "IntervalSeconds": 2,
  "MaxAttempts": 3,
  "BackoffRate": 2.0
}]
```
- **IntervalSeconds**: wait before first retry
- **MaxAttempts**: maximum retry count (0 = no retries)
- **BackoffRate**: multiply interval by this factor on each retry (2× = 2s, 4s, 8s)

### Catch
Route to a different state when retries are exhausted:

```json
"Catch": [{
  "ErrorEquals": ["States.ALL"],
  "Next": "HandleFailure",
  "ResultPath": "$.error"
}]
```
- `States.ALL` catches any error
- `ResultPath` controls where the error detail is injected into the state input

### Common error names
| Error | Meaning |
|---|---|
| `States.ALL` | Matches any error |
| `States.Timeout` | Task exceeded `TimeoutSeconds` |
| `States.TaskFailed` | Task returned an error |
| `States.HeartbeatTimeout` | Activity worker stopped sending heartbeats |
| `Lambda.ServiceException` | Lambda service error |
| `Lambda.TooManyRequestsException` | Lambda throttle |

## Service Integrations

Step Functions Task states can call AWS services directly — no Lambda needed as a glue layer.

### Integration patterns

| Pattern | Behaviour | Use case |
|---|---|---|
| **Request-Response** (default) | Call service, move to next state immediately (don't wait) | Fire-and-forget — trigger a job |
| **Sync (.sync)** | Wait for the job/task to complete before proceeding | ECS tasks, Glue jobs, EMR steps |
| **Wait for Token (.waitForTaskToken)** | Pause execution until an external callback sends a task token | Human approvals, external system callbacks |

### Directly integrated services (selection)
Lambda, DynamoDB, S3, SQS, SNS, ECS, Fargate, Glue, EMR, Athena, Bedrock, SageMaker, API Gateway, EventBridge, and more — over 200 AWS services.

### Wait for Task Token pattern
```text
State Machine pauses
    │
    ├──▶ Sends task token to SQS / Lambda / API
    │
    │   (human reviews, external system processes)
    │
    └──▶ External system calls SendTaskSuccess(taskToken)
              │
         State machine resumes
```
Ideal for: human approval workflows, waiting for a payment gateway, waiting for a third-party API callback.

## EventBridge Fundamentals

EventBridge is a serverless **event bus** that routes events from sources to targets based on rules.

### Event buses

| Bus type | Description |
|---|---|
| **Default event bus** | Receives events from AWS services (EC2 state changes, S3 notifications, CodePipeline, etc.) |
| **Custom event bus** | Receives events from your own applications via `PutEvents` |
| **Partner event bus** | Receives events from SaaS partners — Datadog, Zendesk, GitHub, Stripe, etc. |

### How EventBridge works
```text
Event Source                  EventBridge Bus         Rule              Target
AWS service (EC2 stopped) ──▶ default bus      ──▶  pattern match ──▶ Lambda
Your app (order placed)   ──▶ custom bus       ──▶  pattern match ──▶ SQS / SNS / Step Functions
Stripe (payment event)    ──▶ partner bus      ──▶  pattern match ──▶ Lambda
```

### Event structure
Every EventBridge event is a JSON object:
```json
{
  "version": "0",
  "id": "abc-123",
  "source": "aws.ec2",
  "account": "123456789012",
  "time": "2026-04-18T10:00:00Z",
  "region": "us-east-1",
  "detail-type": "EC2 Instance State-change Notification",
  "detail": { "instance-id": "i-abc", "state": "stopped" }
}
```

## EventBridge Rules

Rules match incoming events and route them to one or more targets.

### Event pattern matching
A rule's event pattern is a JSON filter — an event matches if all specified fields match:

```json
{
  "source": ["aws.ec2"],
  "detail-type": ["EC2 Instance State-change Notification"],
  "detail": {
    "state": ["stopped", "terminated"]
  }
}
```
Matching supports: exact values, prefix matching, numeric ranges, existence checks (`exists: false`), anything-but, and wildcard.

### Targets
A single rule can route to up to **5 targets** simultaneously:
- Lambda, Step Functions, SQS, SNS, Kinesis Data Streams, Firehose
- ECS tasks, CodePipeline, CodeBuild
- API Gateway, EventBridge API destinations (any HTTP endpoint)
- Another EventBridge bus (cross-account / cross-region routing)

### Input transformation
Before delivering to a target, you can reshape the event — extract specific fields, add static values, or restructure the JSON. This avoids the need for a Lambda function just to transform the event shape.

### Scheduled rules (cron)
Rules can fire on a schedule instead of matching an event:
```text
rate(5 minutes)           — every 5 minutes
cron(0 12 * * ? *)        — every day at 12:00 UTC
cron(0 8 ? * MON-FRI *)   — weekdays at 08:00 UTC
```
Use cases: trigger nightly ETL jobs, send weekly digest emails, rotate credentials on a schedule.

## EventBridge Advanced Features

### Schema Registry
EventBridge automatically discovers and registers the schema (structure) of events flowing through your bus. You can browse schemas in the console and generate code bindings (Python, Java, TypeScript) so your application works with typed event objects instead of raw JSON.

### EventBridge Pipes
Pipes create **point-to-point integrations** between a source and a target with optional filtering and enrichment:

```text
Source                Filter      Enrich        Target
SQS / Kinesis / ──▶  (optional) ──▶ Lambda ──▶  Step Functions /
DynamoDB Streams /                               API Gateway /
Kafka / MQ                                       SQS / SNS / etc.
```

Pipes reduce glue code — instead of writing a Lambda to poll SQS, transform records, and forward to Step Functions, a Pipe does this with no code.

### Cross-account / cross-region event routing
You can send events from one account's bus to another account's bus. Pattern: a central event bus in a logging/security account receives events from all organisational accounts for centralised processing.

### Archive and Replay
EventBridge can **archive** all events matching a pattern for a configured retention period. You can then **replay** archived events to a bus — useful for debugging, reprocessing after a bug fix, or testing new consumers against historical traffic.

## EventBridge vs SNS vs SQS

| | EventBridge | SNS | SQS |
|---|---|---|---|
| Model | Event bus with rules | Pub/Sub push | Queue (pull) |
| Filtering | Rich pattern matching on any field | Subscription filter policies on attributes | No filtering |
| Sources | AWS services, SaaS, custom | Application | Application |
| Targets | 20+ AWS service types | SQS, Lambda, HTTP, Email, SMS | One consumer |
| Persistence | Archive (optional) | No | Yes — up to 14 days |
| Use case | Route AWS/SaaS/custom events with complex rules | Fan-out to many subscribers | Buffer work between producer and consumer |

> **Decision guide:**
> - React to AWS service events (EC2 stopped, S3 upload, CodePipeline state) → **EventBridge**
> - Fan-out one event to many subscribers simultaneously → **SNS**
> - Decouple and buffer work; one consumer per message → **SQS**
> - Orchestrate multi-step workflows with branching, retries, and state → **Step Functions**

## Working with Step Functions and EventBridge via boto3

In [ ]:
import boto3
import json

sfn    = boto3.client('stepfunctions', region_name='us-east-1')
eb     = boto3.client('events',        region_name='us-east-1')

In [ ]:
# Define a state machine in Amazon States Language (ASL)
order_workflow = {
    "Comment": "Order fulfilment workflow",
    "StartAt": "ValidateOrder",
    "States": {
        "ValidateOrder": {
            "Type": "Task",
            "Resource": "arn:aws:lambda:us-east-1:123456789012:function:validate-order",
            "Retry": [{"ErrorEquals": ["Lambda.ServiceException"],
                       "IntervalSeconds": 2, "MaxAttempts": 3, "BackoffRate": 2}],
            "Catch": [{"ErrorEquals": ["States.ALL"], "Next": "OrderFailed",
                       "ResultPath": "$.error"}],
            "Next": "CheckInventory"
        },
        "CheckInventory": {
            "Type": "Choice",
            "Choices": [
                {"Variable": "$.inventoryAvailable",
                 "BooleanEquals": True, "Next": "ProcessPayment"}
            ],
            "Default": "OutOfStock"
        },
        "ProcessPayment": {
            "Type": "Task",
            "Resource": "arn:aws:lambda:us-east-1:123456789012:function:process-payment",
            "Next": "NotifyAndReserve"
        },
        "NotifyAndReserve": {
            "Type": "Parallel",       # run both branches simultaneously
            "Branches": [
                {"StartAt": "SendConfirmation", "States": {
                    "SendConfirmation": {"Type": "Task",
                        "Resource": "arn:aws:states:::sns:publish",
                        "Parameters": {"TopicArn": "arn:aws:sns:us-east-1:123456789012:orders",
                                       "Message.$": "$.orderId"},
                        "End": True}}},
                {"StartAt": "ReserveInventory", "States": {
                    "ReserveInventory": {"Type": "Task",
                        "Resource": "arn:aws:states:::dynamodb:updateItem",
                        "Parameters": {"TableName": "Inventory",
                                       "Key": {"itemId": {"S.$": "$.itemId"}},
                                       "UpdateExpression": "SET reserved = reserved + :one",
                                       "ExpressionAttributeValues": {":one": {"N": "1"}}},
                        "End": True}}}
            ],
            "End": True
        },
        "OutOfStock": {"Type": "Fail", "Error": "OutOfStock", "Cause": "Item not available"},
        "OrderFailed": {"Type": "Fail", "Error": "ValidationFailed"}
    }
}

sm = sfn.create_state_machine(
    name='OrderFulfilment',
    definition=json.dumps(order_workflow),
    roleArn='arn:aws:iam::123456789012:role/StepFunctionsRole',
    type='STANDARD'
)
print(f"State machine ARN: {sm['stateMachineArn']}")

In [ ]:
# Start an execution
execution = sfn.start_execution(
    stateMachineArn=sm['stateMachineArn'],
    name='order-ORD-001',
    input=json.dumps({'orderId': 'ORD-001', 'itemId': 'ITEM-42', 'amount': 99.99})
)
print(f"Execution ARN: {execution['executionArn']}")

# Check execution status
status = sfn.describe_execution(executionArn=execution['executionArn'])
print(f"Status: {status['status']}")  # RUNNING, SUCCEEDED, FAILED, TIMED_OUT, ABORTED

In [ ]:
# Create an EventBridge rule — trigger Step Functions when an order is placed
eb.put_rule(
    Name='OnOrderPlaced',
    EventBusName='default',
    EventPattern=json.dumps({
        'source':      ['myapp.orders'],
        'detail-type': ['OrderPlaced'],
        'detail': {
            'status': ['new'],
            'amount': [{'numeric': ['>', 0]}]
        }
    }),
    State='ENABLED',
    Description='Start order fulfilment workflow on new orders'
)

# Add Step Functions as the target
eb.put_targets(
    Rule='OnOrderPlaced',
    EventBusName='default',
    Targets=[{
        'Id': 'StartOrderWorkflow',
        'Arn': sm['stateMachineArn'],
        'RoleArn': 'arn:aws:iam::123456789012:role/EventBridgeStepFunctionsRole',
        'InputTransformer': {
            'InputPathsMap': {'orderId': '$.detail.orderId',
                              'itemId':  '$.detail.itemId',
                              'amount':  '$.detail.amount'},
            'InputTemplate': '{"orderId": "<orderId>", "itemId": "<itemId>", "amount": <amount>}'
        }
    }]
)
print("EventBridge rule and target configured")

In [ ]:
# Scheduled rule — trigger a Lambda nightly at 02:00 UTC
eb.put_rule(
    Name='NightlyDataExport',
    ScheduleExpression='cron(0 2 * * ? *)',   # 02:00 UTC every day
    State='ENABLED'
)

eb.put_targets(
    Rule='NightlyDataExport',
    Targets=[{
        'Id': 'NightlyExportLambda',
        'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:nightly-export',
        'Input': json.dumps({'exportType': 'full', 'destination': 's3://my-exports/'})
    }]
)
print("Scheduled rule created")

In [ ]:
# Publish a custom event to EventBridge
eb.put_events(
    Entries=[{
        'Source':       'myapp.orders',
        'DetailType':   'OrderPlaced',
        'EventBusName': 'default',
        'Detail': json.dumps({
            'orderId': 'ORD-001',
            'itemId':  'ITEM-42',
            'amount':  99.99,
            'status':  'new'
        })
    }]
)
print("Custom event published — EventBridge will route to matching rules")

## Summary

| Concept | Key Takeaway |
|---|---|
| Step Functions | Managed state machine orchestration; moves workflow logic out of application code |
| Amazon States Language | JSON definition of states and transitions |
| Task state | Invoke Lambda, call AWS service directly |
| Choice state | Branch on input data (if/else) |
| Parallel state | Run multiple branches simultaneously; wait for all |
| Map state | Iterate over array; apply sub-workflow to each item |
| Wait state | Pause for duration or until timestamp |
| Standard workflow | Up to 1 year; exactly-once; per state transition pricing; full audit history |
| Express workflow | Up to 5 minutes; at-least-once; high throughput; logs to CloudWatch |
| Retry | Exponential backoff on Task failure; built into state definition |
| Catch | Route to error-handling state when retries exhausted |
| Wait for Task Token | Pause execution until external callback — human approvals, third-party systems |
| .sync integration | Wait for long-running job (ECS, Glue, EMR) to finish before proceeding |
| EventBridge | Serverless event bus; routes events via pattern-matching rules |
| Default bus | AWS service events (EC2, S3, CodePipeline, etc.) |
| Custom bus | Your application's events via PutEvents |
| Partner bus | SaaS partner events (Datadog, Stripe, GitHub, etc.) |
| EventBridge rule | Pattern match → route to up to 5 targets |
| Scheduled rule | cron or rate expression — replace cron jobs |
| Input transformer | Reshape event before delivering to target — no Lambda needed |
| Schema Registry | Auto-discovers event schemas; generates code bindings |
| EventBridge Pipes | Point-to-point source → filter → enrich → target; minimal glue code |
| Archive & Replay | Store events; replay to reprocess after bug fix |
| EB vs SNS vs SQS | EB: route AWS/SaaS events with rules; SNS: fan-out; SQS: buffer queue |